In [0]:
# RevOps CRM Pipeline Project
catalog = "flora_worksplace"
schema = "default"
volume = "revops_crm_data"
file_name = "sales_pipeline.csv"

# Unity Catalog paths

path_volume = f"/Volumes/{catalog}/{schema}/{volume}"

path_table = f"{catalog}.{schema}"

In [0]:
print("Catalog:", catalog)
print("Schema:", schema)
print("Volume:", volume)
print("File:", file_name)
print("Volume Path:", path_volume)

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS flora_workspace.default.revops_crm_data;

In [0]:
# Check files in the RevOps data volume

display(dbutils.fs.ls(path_volume))

In [0]:
%sql
SHOW CATALOGS;

In [0]:
%sql
SHOW SCHEMAS IN flora_workspace;

In [0]:
%sql
SHOW VOLUMES IN flora_workspace.default;

In [0]:
# ==========================================
# REVOPS CRM PIPELINE & FORECAST PROJECT
# ==========================================

catalog = "flora_workspace"
schema = "default"
volume = "revops_crm_data"

file_name = "sales_pipeline.csv"

path_volume = f"/Volumes/{catalog}/{schema}/{volume}"

print("Volume Path:", path_volume)

In [0]:
display(dbutils.fs.ls(path_volume))

In [0]:

df_sales_pipeline = spark.read.csv(
    f"{path_volume}/sales_pipeline.csv",
    header=True,
    inferSchema=True
)

display(df_sales_pipeline)

In [0]:
# Check column names and data types

df_sales_pipeline.printSchema()

In [0]:
# Count sales opportunities

df_sales_pipeline.count()

In [0]:
# Check null values in each column

from pyspark.sql import functions as F

df_sales_pipeline.select([
    F.sum(F.col(column).isNull().cast("int")).alias(column)
    for column in df_sales_pipeline.columns
]).display()

In [0]:
# Show the columns available for analysis

print(df_sales_pipeline.columns)

In [0]:
# Reload the CRM sales pipeline dataset

df_sales_pipeline = spark.read.csv(
    "/Volumes/flora_workspace/default/revops_crm_data/sales_pipeline.csv",
    header=True,
    inferSchema=True
)

print("Rows:", df_sales_pipeline.count())
print("Columns:", df_sales_pipeline.columns)

In [0]:
df_sales_pipeline.printSchema()

In [0]:
df_sales_pipeline.printSchema()

In [0]:
# Create a permanent Delta table from the PySpark DataFrame

table_name = "flora_workspace.default.sales_pipeline"

df_sales_pipeline.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(table_name)

print(f"Permanent table created successfully: {table_name}")

In [0]:
# Load the CRM sales pipeline dataset

df_sales_pipeline = spark.read.csv(
    "/Volumes/flora_workspace/default/revops_crm_data/sales_pipeline.csv",
    header=True,
    inferSchema=True
)

print("Rows:", df_sales_pipeline.count())
print("Columns:", df_sales_pipeline.columns)

In [0]:
# Make the Python DataFrame available to SQL

df_sales_pipeline.createOrReplaceTempView("sales_pipeline_raw")

print("SQL view created successfully")

In [0]:
%sql
SELECT
    deal_stage,
    COUNT(*) AS opportunity_count
FROM flora_workspace.default.sales_pipeline
GROUP BY deal_stage
ORDER BY opportunity_count DESC;

In [0]:
%sql
SELECT
    sales_agent,
    COUNT(*) AS opportunity_count,
     CONCAT('$', FORMAT_NUMBER(SUM(close_value), 0)) AS amount
     FROM flora_workspace.default.sales_pipeline
GROUP BY sales_agent
ORDER BY amount DESC;

In [0]:
%sql
SELECT
    product,
    COUNT(*) AS opportunity_count,
     CONCAT('$', FORMAT_NUMBER(SUM(close_value), 0)) AS amount
FROM flora_workspace.default.sales_pipeline
GROUP BY product
ORDER BY amount DESC;

In [0]:
%sql
SELECT
    CONCAT('$', FORMAT_NUMBER(SUM(close_value)/COUNT(opportunity_id),00)) AS average_deal_size
    FROM flora_workspace.default.sales_pipeline


In [0]:
%sql
SELECT
    deal_stage,
    COUNT(*) AS opportunity_count
FROM flora_workspace.default.sales_pipeline
GROUP BY deal_stage
HAVING deal_stage not in ('Won','Lost')

In [0]:
%sql

WITH account_opportunities AS (
    SELECT
        account,
        COUNT(*) AS open_opportunity_count
    FROM flora_workspace.default.sales_pipeline
    WHERE deal_stage NOT IN ('Won', 'Lost')
      AND account IS NOT NULL
    GROUP BY account
),

total_open AS (
    SELECT
        SUM(open_opportunity_count) AS total_open_opportunities
    FROM account_opportunities
)

SELECT
    account,
    open_opportunity_count,

    CONCAT(
        ROUND(
            100.0 * open_opportunity_count / total_open_opportunities,
            2
        ),
        '%'
    ) AS opportunity_share_pct,

    RANK() OVER (
        ORDER BY open_opportunity_count DESC
    ) AS opportunity_rank

FROM account_opportunities
CROSS JOIN total_open

ORDER BY opportunity_rank;

In [0]:
%sql

WITH rep_opportunities AS (
    SELECT
        sales_agent,
        COUNT(*) AS open_opportunity_count
    FROM flora_workspace.default.sales_pipeline
    WHERE deal_stage NOT IN ('Won', 'Lost')
      AND sales_agent IS NOT NULL
    GROUP BY sales_agent
),

total_open AS (
    SELECT
        SUM(open_opportunity_count) AS total_open_opportunities
    FROM rep_opportunities
)

SELECT
    sales_agent,
    open_opportunity_count,

    CONCAT(
        ROUND(
            100.0 * open_opportunity_count / total_open_opportunities,
            2
        ),
        '%'
    ) AS opportunity_share_pct,

    RANK() OVER (
        ORDER BY open_opportunity_count DESC
    ) AS opportunity_rank

FROM rep_opportunities
CROSS JOIN total_open

ORDER BY opportunity_rank;

In [0]:
%sql

SELECT
    sales_agent,

    COUNT(*) AS won_opportunities,

    ROUND(
        AVG(
            DATEDIFF(close_date, engage_date)
        ),
        1
    ) AS avg_sales_cycle_days

FROM flora_workspace.default.sales_pipeline

WHERE deal_stage = 'Won'
  AND engage_date IS NOT NULL
  AND close_date IS NOT NULL

GROUP BY sales_agent

ORDER BY avg_sales_cycle_days DESC;